In [46]:
!pip -q install gradio pandas huggingface_hub "transformers<5" torch accelerate

In [47]:
import os, getpass
def get_hf_token():
    # 1) Colab Secrets (only works in Google Colab)
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    # 2) Environment variable (works locally/Jupyter)
    token = os.getenv("HF_TOKEN")
    if token:
        return token

    # 3) Manual input (one-time)
    return getpass.getpass("Paste your HF_TOKEN (input hidden): ")

HF_TOKEN = get_hf_token()
print("HF_TOKEN loaded ✅" if HF_TOKEN else "HF_TOKEN missing ❌")

HF_TOKEN loaded ✅


In [48]:
import json
import pandas as pd
import gradio as gr
from huggingface_hub import InferenceClient

In [49]:
SCHEMAS = {
    "Support Tickets": ["ticket_id","category","priority","message","resolution"],
    "Fintech Transactions": ["tx_id","amount","currency","status","failure_reason"],
}

STYLES = {
    "Normal": "realistic, clean examples",
    "Noisy": "typos, slang, short messages, real-world ambiguity",
    "Edge cases": "rare failure scenarios and tricky cases",
}

In [50]:
def make_prompt(dataset, n, style):
    fields = SCHEMAS[dataset]
    return f"""
Generate {n} rows of {dataset} as VALID JSON only.

Rules:
- Output must be a JSON array with exactly {n} objects.
- Each object must have EXACTLY these keys: {fields}
- Make it {STYLES[style]}.
- IDs must be unique.
Return ONLY JSON.
""".strip()

In [51]:
client = InferenceClient(provider="hf-inference", token=HF_TOKEN)
def generate(dataset, n_rows, style, model_id):
    prompt = make_prompt(dataset, int(n_rows), style)

    text = client.text_generation(
        model=model_id,
        prompt=prompt,
        max_new_tokens=800,
        temperature=0.8,
        top_p=0.95,
        do_sample=True,
    )

    data = json.loads(text)  # because we forced "JSON only"
    df = pd.DataFrame(data)[SCHEMAS[dataset]]

    path = "/content/synthetic.csv"
    df.to_csv(path, index=False)
    return df, path

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 🧪 Simple Synthetic Data Generator")

    dataset = gr.Dropdown(list(SCHEMAS.keys()), value="Support Tickets", label="Dataset")
    style = gr.Dropdown(list(STYLES.keys()), value="Normal", label="Style")
    n_rows = gr.Slider(5, 50, value=10, step=1, label="Rows")

    model_id = gr.Textbox(value="HuggingFaceH4/zephyr-7b-beta", label="HF Model ID")
    
    btn = gr.Button("Generate ✅")
    table = gr.Dataframe(interactive=False, label="Preview")
    file = gr.File(label="Download CSV")

    btn.click(generate, inputs=[dataset, n_rows, style, model_id], outputs=[table, file])

demo.launch(share=True)